# Shopify Product ID Checker

**Target:** Halaman produk (`/products/{slug}`) — bukan blog, page, atau collection.

**Cara ambil ID (prioritas):**
1. **Endpoint `.json`** → `https://domain.com/products/{slug}.json` → field `product.id`
2. **Fallback `var __st`** → parsing HTML, cari `{"p": "product", "rid": <id>}`
3. **Fallback `ShopifyAnalytics`** → cari `"resourceType":"product"` + `"resourceId"`

**Output:** `shopify_product_ids.csv`

**CMS URL:** `https://admin.shopify.com/store/{store}/products/{id}`

---

> **Catatan:** Metode `.json` endpoint lebih reliable karena tidak perlu scraping HTML,
> langsung dapat response JSON bersih. Beberapa store mungkin disable endpoint ini
> (password-protected atau custom config) — kalau itu terjadi, fallback HTML otomatis aktif.

In [ ]:
from google.colab import files

# ── Input Store Name ─────────────────────────────────────────
STORE_NAME = input("Masukkan Shopify store name (contoh: sme-brother): ").strip()

if not STORE_NAME:
    raise ValueError("Store name tidak boleh kosong!")

ADMIN_BASE = f"https://admin.shopify.com/store/{STORE_NAME}"
print(f"\n Store name     : {STORE_NAME}")
print(f" Admin base URL : {ADMIN_BASE}/")

# ── Upload File URL ──────────────────────────────────────────
print("\n Silakan upload file .txt berisi URL halaman produk Shopify...")
print("   Format: satu URL per baris, contoh:")
print("   https://smebrother.com/products/hong-kong-accounting-香港公司會計做賬")
uploaded = files.upload()

if not uploaded:
    raise ValueError("Tidak ada file yang diupload!")

INPUT_FILE  = list(uploaded.keys())[0]
OUTPUT_FILE = "shopify_product_ids.csv"

print(f"\n File diterima  : {INPUT_FILE}")
print(f" Output file    : {OUTPUT_FILE}")

In [ ]:
# ============================================================
# Shopify Product ID Checker
# Target   : /products/{slug} — halaman detail produk
# ID source: (1) .json endpoint  (2) var __st  (3) ShopifyAnalytics
# CMS URL  : https://admin.shopify.com/store/{store}/products/{id}
# Output   : url, product_slug, shopify_product_id, cms_url, status
# Saves    : real-time ke CSV
# ============================================================

import re
import json
import time
import pathlib
import requests
import pandas as pd
from urllib.parse import urlparse, unquote
from datetime import datetime

# ── Config ──────────────────────────────────────────────────
DELAY_SECONDS   = 5
REQUEST_TIMEOUT = 30

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
)

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.8,id;q=0.7,zh;q=0.6",
    "Connection": "keep-alive",
})

# ── Logger ───────────────────────────────────────────────────
def log(level, msg):
    icons = {
        "INFO":  "i ",
        "OK":    "OK",
        "WARN":  "! ",
        "ERROR": "X ",
        "SKIP":  ">>",
        "SAVE":  "S ",
        "DONE":  "* ",
        "DELAY": "~ ",
        "LINK":  "@ ",
        "JSON":  "{}",
    }
    ts   = datetime.now().strftime("%H:%M:%S")
    icon = icons.get(level, "  ")
    print(f"[{ts}] {icon} [{level}] {msg}")

# ── Validation ───────────────────────────────────────────────
def is_product_url(url):
    """
    Valid: /products/{slug}  (tepat 2 bagian path)
    Invalid: /products/{slug}/variants/xxx  (lebih dari 2 bagian)
    """
    try:
        path  = unquote(urlparse(url).path).strip("/")
        parts = path.split("/")
        return len(parts) == 2 and parts[0] == "products" and parts[1] != ""
    except Exception:
        return False

# ── Slug Extractor ───────────────────────────────────────────
def extract_product_slug(url):
    try:
        path  = unquote(urlparse(url).path)
        parts = path.strip("/").split("/")
        if len(parts) >= 2 and parts[0] == "products":
            return parts[1]
    except Exception:
        pass
    return None

# ── Build .json URL ──────────────────────────────────────────
def build_json_url(url):
    """
    Shopify punya endpoint JSON bawaan untuk product:
    https://domain.com/products/{slug}.json
    Mengembalikan data produk lengkap termasuk 'id'.
    """
    parsed = urlparse(url)
    # path sudah dalam bentuk /products/{slug} — tinggal tambah .json
    json_path = parsed.path.rstrip("/") + ".json"
    return f"{parsed.scheme}://{parsed.netloc}{json_path}"

# ── ID Extractor dari JSON endpoint ─────────────────────────
def fetch_product_id_via_json(url):
    """
    Metode PRIMER: ambil product ID via endpoint .json
    Return: (product_id: int | None, method: str)
    """
    json_url = build_json_url(url)
    try:
        resp = SESSION.get(
            json_url,
            timeout=REQUEST_TIMEOUT,
            headers={"Accept": "application/json"}
        )
        if resp.status_code == 200:
            data = resp.json()
            product = data.get("product", {})
            pid = product.get("id")
            if isinstance(pid, int):
                return pid, "json_endpoint"
            if isinstance(pid, str) and pid.isdigit():
                return int(pid), "json_endpoint"
        # 404 = endpoint disabled atau produk tidak ada
        return None, f"json_http_{resp.status_code}"
    except Exception as e:
        return None, f"json_error ({e})"

# ── ID Extractor dari HTML ────────────────────────────────────
def extract_product_id_from_html(html):
    """
    Metode FALLBACK: parse HTML untuk cari product ID
    Coba 3 pola berbeda sesuai versi Shopify theme.
    Return: (product_id: int | None, method: str)
    """
    # Metode 1: var __st = {...}
    # Shopify inject ini di hampir semua halaman
    m = re.search(r"var\s+__st\s*=\s*(\{.*?\});", html, flags=re.DOTALL)
    if m:
        try:
            payload = json.loads(m.group(1))
            if payload.get("p") == "product":
                rid = payload.get("rid")
                if isinstance(rid, int):
                    return rid, "__st"
                if isinstance(rid, str) and rid.isdigit():
                    return int(rid), "__st"
        except json.JSONDecodeError:
            pass

    # Metode 2: ShopifyAnalytics resourceId
    m2 = re.search(
        r'"resourceType"\s*:\s*"product"[^}]*?"resourceId"\s*:\s*(\d+)',
        html, flags=re.DOTALL
    )
    if m2:
        return int(m2.group(1)), "shopify_analytics"

    # Metode 3: var meta = {...} (theme Debut & sejenis)
    m3 = re.search(
        r'var\s+meta\s*=\s*\{.*?"pageType"\s*:\s*"product".*?"resourceId"\s*:\s*(\d+)',
        html, flags=re.DOTALL
    )
    if m3:
        return int(m3.group(1)), "var_meta"

    return None, "not_found"

# ── CMS URL Builder ──────────────────────────────────────────
def build_cms_url(store_name, product_id):
    return f"https://admin.shopify.com/store/{store_name}/products/{product_id}"

# ── Main Fetcher ─────────────────────────────────────────────
def fetch_product_id(url):
    result = {
        "url":                url,
        "product_slug":       None,
        "shopify_product_id": None,
        "id_method":          None,
        "cms_url":            None,
        "status":             None,
    }

    if not is_product_url(url):
        result["status"] = "skipped (bukan product URL)"
        log("SKIP", f"URL bukan /products/{{slug}}: {url}")
        return result

    result["product_slug"] = extract_product_slug(url)

    # ── Metode 1: .json endpoint ────────────────────────────
    log("JSON", f"Mencoba .json endpoint: {build_json_url(url)}")
    pid, method = fetch_product_id_via_json(url)

    if pid:
        log("OK",   f"Slug     : {result['product_slug']}")
        log("OK",   f"Product ID: {pid}  (via {method})")
        result["shopify_product_id"] = pid
        result["id_method"]          = method
        result["cms_url"]            = build_cms_url(STORE_NAME, pid)
        result["status"]             = "ok"
        log("LINK", f"CMS URL  : {result['cms_url']}")
        return result

    # ── Metode 2: Parse HTML ─────────────────────────────────
    log("WARN", f".json endpoint tidak tersedia ({method}), fallback ke HTML scraping...")
    try:
        resp = SESSION.get(url, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
    except requests.exceptions.HTTPError as e:
        result["status"] = f"error HTTP {e.response.status_code}"
        log("ERROR", result["status"])
        return result
    except Exception as e:
        result["status"] = f"error ({e})"
        log("ERROR", result["status"])
        return result

    pid, method = extract_product_id_from_html(resp.text)

    if pid:
        log("OK",   f"Slug      : {result['product_slug']}")
        log("OK",   f"Product ID: {pid}  (via {method})")
        result["shopify_product_id"] = pid
        result["id_method"]          = method
        result["cms_url"]            = build_cms_url(STORE_NAME, pid)
        result["status"]             = "ok"
        log("LINK", f"CMS URL  : {result['cms_url']}")
    else:
        result["status"] = "error (ID tidak ditemukan di HTML)"
        log("ERROR", result["status"])

    return result

# ── CSV Writer ───────────────────────────────────────────────
COLUMNS = ["url", "product_slug", "shopify_product_id", "id_method", "cms_url", "status"]

def save_row(row, output_file, is_first):
    df_row = pd.DataFrame([row], columns=COLUMNS)
    df_row.to_csv(
        output_file,
        mode="w" if is_first else "a",
        index=False,
        header=is_first,
    )
    log("SAVE", f"Baris disimpan ke {output_file}")

# ── Read Input ───────────────────────────────────────────────
raw_urls = pathlib.Path(INPUT_FILE).read_text(encoding="utf-8").splitlines()
urls = [u.strip() for u in raw_urls if u.strip() and not u.strip().startswith("#")]

log("INFO", f"Store    : {STORE_NAME}")
log("INFO", f"Ditemukan {len(urls)} URL produk")
print("=" * 60)
log("SAVE", f"Output CSV diinisiasi: {OUTPUT_FILE}")

# ── Process ──────────────────────────────────────────────────
results      = []
ok_count     = 0
error_count  = 0
skip_count   = 0

for idx, url in enumerate(urls, start=1):
    print(f"\n-- [{idx}/{len(urls)}] " + "-" * 42)
    log("INFO", f"URL      : {url}")

    row = fetch_product_id(url)
    results.append(row)
    save_row(row, OUTPUT_FILE, is_first=(idx == 1))

    s = row["status"] or ""
    if s == "ok":
        ok_count += 1
    elif s.startswith("skipped"):
        skip_count += 1
    else:
        error_count += 1

    if idx < len(urls):
        log("DELAY", f"Menunggu {DELAY_SECONDS} detik...")
        time.sleep(DELAY_SECONDS)

# ── Summary ──────────────────────────────────────────────────
print("\n" + "=" * 60)
log("DONE", f"Selesai! Total: {len(results)} | OK: {ok_count} | Gagal: {error_count} | Skip: {skip_count}")
log("SAVE", f"Hasil tersimpan di: {OUTPUT_FILE}")

In [ ]:
import pandas as pd
import os

# Hardcoded agar tidak bergantung pada variabel dari cell lain
OUTPUT_FILE = "shopify_product_ids.csv"

if os.path.exists(OUTPUT_FILE):
    df = pd.read_csv(OUTPUT_FILE)
    print(f"Total rows  : {len(df)}")
    print(f"Berhasil    : {(df['status'] == 'ok').sum()}")
    print(f"Gagal       : {df['status'].str.startswith('error').sum()}")
    print(f"Skip        : {df['status'].str.startswith('skipped').sum()}")
    print()
    display(df)
else:
    print("File output belum tersedia.")